In [ ]:
# connect to GDrive
from google.colab import drive
drive.mount('[REDACTED]')

### Import Libraries

In [ ]:
!pip install pyLDAvis

In [ ]:
# import modules and libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import ngrams, FreqDist
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import string
import os
import pandas as pd
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from gensim import corpora
from gensim.models import LdaModel
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
import pyLDAvis
import pyLDAvis.gensim
import matplotlib.pyplot as plt
import openpyxl
import pandas as pd
import requests
from io import BytesIO
import seaborn as sns
import matplotlib.pyplot as plt

### Utility Functions

In [ ]:
# read text files
def read_text_files(file_path):
  with open(file_path, 'r') as file:
    return file.read()

# tokenize the text
def tokenize_text(text):
  text = word_tokenize(text)
  return text

# remove stop words
def remove_stopwords(word_tokens):
  stop_words = set(stopwords.words('english'))

  filtered_sentence = []

  for w in word_tokens:
      if w not in stop_words:
          filtered_sentence.append(w)
  return filtered_sentence

# normalize the text to lower case
def case_normalization(text):
  return [t.lower() for t in text]

# remove punctuations
def remove_punctuation(text):
  res = [re.sub(r'[^\w\s]', '', token) for token in text if re.sub(r'[^\w\s]', '', token)]
  filtered_text = []
  for w in res:
      if len(w.strip()) > 1:
          filtered_text.append(w)
  return filtered_text


# def lemmatize_tokens(text):
#   lemmatizer = WordNetLemmatizer()
#   return [lemmatizer.lemmatize(token) for token in text]

# get the most frequent ngrams
def most_freq_ngrams(text, n):
  ngrams = nltk.ngrams(text, n)
  ngram_fdist = nltk.FreqDist(ngrams)
  return ngram_fdist

# generate a word cloud
def generate_wordcloud(ngram_fdist):
    # Create a dictionary of n-grams and their frequencies
    n_grams_dict = { ' '.join(gram): freq for gram, freq in ngram_fdist.items() }

    # Create a word cloud
    wordcloud = WordCloud(width=800, height=400, background_color='white').[REDACTED](n_grams_dict)

    # Display the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  # Hide the axes
    plt.show()

# find all text files in a directory and add them to a list
def find_texts(directory):
    text_files = []
    # Traverse the directory and subdirectories
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".txt"):
                text_files.append(os.path.join(root, file))
    return text_files

# given dictionary d1 and d2, update d1 with words and counts from d2
def update_dict(d1, d2):
  for k in d2.keys():
    if k in d1.keys():
      d1[k] += d2[k]
    else:
      d1[k] = d2[k]
  return d1

### [REDACTED] - Traditional Topic Modelling

In [ ]:
# read in [REDACTED] content
dir = ['[REDACTED]']
text_files = []
for d in dir:
  text_files = text_files + find_texts(d)
len(text_files), text_files[:2]

In [ ]:
# process the files and add them to a list
clean_corpus = []
for f in text_files:
  text = read_text_files(f)
  text = tokenize_text(text)
  text = case_normalization(text)
  text = remove_stopwords(text)
  text = remove_punctuation(text)
  # text = lemmatize_tokens(text)
  clean_corpus.append(text)

In [ ]:
# Build the bigram model
bigram = gensim.models.Phrases(clean_corpus, min_count=5, threshold=100) # higher threshold fewer phrases.

# Faster way to get a sentence clubbed as a bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)




In [ ]:
# make bigrams
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]


In [ ]:
# make bigrams from the cleaned/preprocessed corpus
clean_corpus = make_bigrams(clean_corpus)


### Topic Modelling

In [ ]:
# Creating document-term matrix
dictionary = corpora.Dictionary(clean_corpus)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in clean_corpus]

In [ ]:
# LDA model
lda = LdaModel(doc_term_matrix,  id2word = dictionary,
               num_topics=7,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha='auto',
                                           per_word_topics=True)

# Results
print(lda.print_topics(num_topics=7, num_words=3))
import pyLDAvis
import pyLDAvis.gensim  # don't skip this
import matplotlib.pyplot as plt
%matplotlib inline
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda, doc_term_matrix, dictionary)
vis

In [ ]:
print('\nPerplexity: ', lda.log_perplexity(doc_term_matrix))  # a measure of how good the model is. lower the better.


In [ ]:

# Set num_words to the length of the dictionary to get all words in each topic
num_words = len(dictionary)

# Extract words and probabilities for each topic
topics_data = []

for topic_id, topic_words in lda.show_topics(formatted=False, num_words=num_words):
    for word, prob in topic_words:
        topics_data.append({'Cluster': topic_id, 'Word': word, 'Probability': prob})

# Convert to DataFrame for easier viewing
df_topic_words = pd.DataFrame(topics_data)
print(df_topic_words.head())


In [ ]:
df_topic_words.to_csv('[REDACTED]')

### Compare clusters with manually identified taxonomies

In [ ]:
# read in manually identified taxonomies - words and counts

spreadsheetId = "[REDACTED]" # Please set your Spreadsheet ID.
url = "[REDACTED]" + spreadsheetId
res = requests.get(url)
data = BytesIO(res.content)
xlsx = openpyxl.load_workbook(filename=data)
xlsx.sheetnames

In [ ]:
# List of sheet names
sheet_names = ['Data Monetization',
 'Case Study',
 'Standards',
 'User Experience',
 'Industry',
 'Business Objectives',
 'Analytics',
 'Data Policy',
 'Compliance',
 'Communication',
 'Talent Development',
 'Process',
 'Organization',
 'Data Privacy & Security',
 'Data Stewardship',
 'Techniques, Technologies, Tools',
 'Data Lands[REDACTED]e',
 'Data Management',
 'Data Quality',
 'Data Strategy']  # Replace with your actual sheet names

# Load data from each sheet and add a label column with the sheet name
data_frames = []
for sheet in sheet_names:
    df = pd.read_excel(data, sheet_name=sheet)
    df['Label'] = sheet  # Add a new column with the sheet name as label
    data_frames.append(df)

# Concatenate all data frames into a single data frame
combined_df = pd.concat(data_frames, ignore_index=True)

# Print or use the resulting DataFrame
print(combined_df.head())

In [ ]:
combined_df.to_csv('[REDACTED]')

In [ ]:
# align the format of bigrams with words in topics
combined_df['words'] = combined_df['words'].apply(lambda x: x.replace("('", "").replace("', '", "_").replace("')", ""))

In [ ]:
# read the manually approved bigrams
df_yes = pd.read_excel(data, sheet_name='Yess')
df_yes.head()

In [ ]:
# align the format with words in topics
df_yes['words'] = df_yes['words'].apply(lambda x: x.replace("('", "").replace("', '", "_").replace("')", ""))
df_yes.head()

In [ ]:
# filter topic words bigrams based on Yes's
df_topic_words_bigrams = df_topic_words[df_topic_words['Word'].str.contains('_')]
df_topic_words_bigrams.head()

In [ ]:
# filter topic words bigrams based on Yes's
df_topic_words_bigrams = df_topic_words_bigrams[df_topic_words_bigrams['Word'].isin(df_yes['words'])]
df_topic_words_bigrams.head()

In [ ]:
df_topic_words_bigrams.shape

In [ ]:
df_topic_words_bigrams.describe()

In [ ]:
# keep the bigrams with top 75 percentile probability
df_topic_words_bigrams[df_topic_words_bigrams['Probability']>= 1.700672e-05].shape

In [ ]:
# top 75 % bigrams in each cluster
top_75 = df_topic_words_bigrams[df_topic_words_bigrams['Probability']>= 1.700672e-05]

In [ ]:
# number of bigrams in each cluster
top_75.groupby('Cluster').count()

In [ ]:
# drop duplicates
combined_df = combined_df.drop_duplicates(subset='words')
combined_df.shape

In [ ]:
# merge with taxonomies
topic_taxonomies = combined_df.merge(top_75, left_on='words', right_on='Word', how='right')
topic_taxonomies.shape


In [ ]:
topic_taxonomies.head()

In [ ]:
# taxonomies in each topic
topic_taxonomies[['Label','Cluster']].groupby('Cluster').agg(lambda x: x.unique().tolist())

In [ ]:
# Count the occurrences of each taxonomy label per cluster
df = topic_taxonomies[['Label','Cluster', 'Word']]
label_counts = df.groupby(['Cluster', 'Label']).size().reset_index(name='count')

# Calculate the total words per cluster
cluster_totals = df.groupby('Cluster').size().reset_index(name='total')

# Merge to calculate percentages
label_counts = label_counts.merge(cluster_totals, on='Cluster')
label_counts['percentage'] = (label_counts['count'] / label_counts['total']) * 100
label_counts['percentage'] = label_counts['percentage'].round(2)
label_counts.head()


In [ ]:
# Sort by cluster and percentage to get the top taxonomy labels in each cluster
top_taxonomies = label_counts.sort_values(['Cluster', 'percentage'], ascending=[True, False])

# Select the top taxonomy label per cluster (adjust as needed)
top_taxonomies = top_taxonomies.groupby('Cluster').head(1).reset_index(drop=True)



In [ ]:
top_taxonomies

### Bar Chart - Top Taxonomy Label In Each Cluster

In [ ]:


# Set plot size
plt.figure(figsize=(10, 6))

# Plot top taxonomy label per cluster
sns.barplot(
    data=top_taxonomies,
    x='Cluster',  # Cluster on x-axis
    y='percentage',  # Percentage on y-axis
    hue='Label',  # Taxonomy label as hue
    dodge=False,  # Combine the bars for each cluster
    palette='tab10'  # Color palette
)

# Add title and labels
plt.title('Top Taxonomy Label by Percentage in Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Percentage')
plt.legend(title='Taxonomy Label', bbox_to_anchor=(1.05, 1), loc='upper left')  # Move legend outside
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()


### Heatmap - Distribution of Texonomy Labels Across Topics

In [ ]:


# Pivot the data for heatmap
heatmap_data = label_counts.pivot(index='Cluster', columns='Label', values='percentage').fillna(0)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', cbar_kws={'label': 'Percentage'})
plt.title('Taxonomy Label Distribution Heatmap Across Clusters')
plt.xlabel('Taxonomy Label')
plt.ylabel('Cluster')
plt.show()


In [ ]:
# Sort by cluster and percentage to get the top taxonomy labels in each cluster
top_taxonomies = label_counts.sort_values(['Cluster', 'percentage'], ascending=[True, False])

# Select the top 3 taxonomy labels per cluster (adjust as needed)
top_taxonomies = top_taxonomies.groupby('Cluster').head(2).reset_index(drop=True)


### Bar chart - Top 2 Taxonomy Labels in Each Cluster (Indivvidual)

In [ ]:


# Set plot size
plt.figure(figsize=(12, 8))

# Use seaborn to create a bar plot with clusters on the x-axis and taxonomy labels on the y-axis
sns.barplot(
    data=top_taxonomies,
    x='Cluster',  # Cluster on the x-axis
    y='percentage',  # Percentage on the y-axis
    hue='Label',  # Taxonomy labels as hue to distinguish between them
    dodge=True,  # Separate bars for each taxonomy label within the same cluster
    palette='tab10'  # Color palette
)

# Add title and labels
plt.title('Taxonomy Labels by Percentage for Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Percentage')
plt.legend(title='Taxonomy Label', bbox_to_anchor=(1.05, 1), loc='upper left')  # Move legend outside
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()


### Bar chart - Top 2 Taxonomy Labels in Each Cluster (Stacked)

In [ ]:


# Set plot size
plt.figure(figsize=(12, 8))

# Use seaborn to create a bar plot with clusters on the x-axis and taxonomy labels on the y-axis
sns.barplot(
    data=top_taxonomies,
    x='Cluster',  # Cluster on the x-axis
    y='percentage',  # Percentage on the y-axis
    hue='Label',  # Taxonomy labels as hue to distinguish between them
    dodge=False,  # Keep bars for the same cluster close to each other
    palette='tab10'  # Color palette
)

# Add title and labels
plt.title('Taxonomy Labels by Percentage for Each Cluster')
plt.xlabel('Cluster')
plt.ylabel('Percentage')
plt.legend(title='Taxonomy Label', bbox_to_anchor=(1.05, 1), loc='upper left')  # Move legend outside
plt.tight_layout()  # Adjust layout to prevent clipping
plt.show()
